## XBRL US API - Python example  
This sample Python code queries the XBRL US Public Filings Database; it is based on a notebook created by [Ties de Kok](https://www.tiesdekok.com).
### Authenticate for access token 
Run the cell below, then type your XBRL US Web account email, account password, Client ID, and secret (get these from https://xbrl.us/access-token), pressing the Enter key on the keyboard after each entry.

XBRL US limits records returned for a query to improve efficiency; this script loops to collect all data from the Public Filings Database for a query. **Non-members might not be able to return all data for a query** - join XBRL US for comprehensive access - https://xbrl.us/join.

In [ ]:
import os, re, sys, json
import requests
import pandas as pd
from IPython.display import display, HTML
import numpy as np
import getpass
from datetime import datetime
import urllib
from urllib.parse import urlencode
api = input('Enter api to use ("api" or other) ')
baseurl = 'https://' + api + '.xbrl.us/'

class tokenInfoClass:
    access_token = None
    refresh_token = None
    email = None
    username = None
    client_id = None
    client_secret = None
    authurl = baseurl + 'oauth2/token'
    headers = {"Content-Type": "application/x-www-form-urlencoded"}
	
def refresh(info):
    refresh_auth = {
                'client_id': info.client_id, 
				'client_secret' : info.client_secret, 
				'grant_type' : 'refresh_token', 
				'platform' : 'ipynb', 
				'refresh_token' : info.refresh_token 
                }
    refreshres = requests.post(info.authurl, data=refresh_auth, headers=info.headers)
    refresh_json = refreshres.json()
    info.access_token = refresh_json['access_token']
    info.refresh_token = refresh_json['refresh_token']
    print('Your access token(%s) is refreshed for 60 minutes. If it expires again, run this cell to generate a new token and continue to use the query cells below.' % (info.access_token))
    return info	

tokenInfo = tokenInfoClass()

tokenInfo.email = input('Enter your XBRL US Web account email: ')
tokenInfo.password = getpass.getpass(prompt='Password: ')
tokenInfo.client_id = getpass.getpass(prompt='Client ID: ')
tokenInfo.client_secret = getpass.getpass(prompt='Secret: ')

body_auth = {'username' : tokenInfo.email, 
            'client_id': tokenInfo.client_id, 
            'client_secret' : tokenInfo.client_secret, 
            'password' : tokenInfo.password, 
            'grant_type' : 'password', 
            'platform' : 'ipynb' }

#print(body_auth)

payload = urlencode(body_auth)
res = requests.request("POST", tokenInfo.authurl, data=payload, headers=tokenInfo.headers)
auth_json = res.json()

if 'error' in auth_json:
    print("\n\nThere was a problem generating the access token: %s.  Run the first cell again and enter the credentials." % (auth_json['error_description']))
else:
    tokenInfo.access_token = auth_json['access_token']
    tokenInfo.refresh_token = auth_json['refresh_token']
    print ("\n\nYour access token expires in 60 minutes. After it expires, it should be regenerated automatically.  If not, run the cell rerun the first query cell. \n\nFor now, skip ahead to the section 'Make a Query'.")
	
#print(vars(tokenInfo))
print('\n\naccess token: ' + tokenInfo.access_token + ' refresh token: ' + tokenInfo.refresh_token)

### Make a query 
After the access token confirmation appears above, you can modify the query below, then use the **_Cell >> Run_** menu option from the cell **immediately below this text** to run the entire query for results.

The sample results are from 10+ years of data for companies in an SIC code, and may take several minutes to recreate. **To test for results quickly, modify the _params_** to comment out report.sic-code and uncomment entity.cik and period.fiscal-year so the search runs for several companies across a few years.
  
Refer to XBRL API documentation at https://xbrlus.github.io/xbrl-api/#/Facts/getFactDetails for other endpoints and parameters to filter and return. 

In [14]:
# Define the parameters of the query - this query returns all of the most-recent
# reported fiscal year values for all years as defined below (XBRL_Elements)
# in companies reporting with SIC code 2080

endpoint = 'assertion'
XBRL_Elements = [
    'DQC.US.0194.10621',
    'DQC.US.0194.10636',
    'DQC.US.0195.10622',
    'DQC.US.0195.10623',
    'DQC.US.0195.10624',
    'DQC.US.0195.10625',
    'DQC.US.0195.10626',
    'DQC.US.0195.10627',
    'DQC.US.0196.10628',
    'DQC.US.0196.10629',
    'DQC.US.0196.10630',
    'DQC.US.0196.10631',
    'DQC.US.0196.10632',
    'DQC.US.0196.10633',
    'DQC.US.0197.10634',
    'DQC.US.0196.10635',
    ]
report_year = [
    '2023',       
    '2024'
    ]
fields = [ 
     # this is the list of the characteristics of the data being returned by the query
    'report.entry-url',
    'assertion.code.sort(ASC)',
    'report.base-taxonomy',
    'report.document-type',
    'assertion.run-date',
    'report.accepted-timestamp.sort(DESC)',
    'report.accession',
    'entity.code',
    'entity.name',
    'assertion.type',
    #'assertion.detail',
    'assertion.limit()'
    ]

# Set unique rows as True of False (True drops any duplicate rows)
unique = True

# Limit the number of rows displayed by the notebook (does not impact the data frame)
rows_to_display = 100 # Set as '' to display all rows in the notebook

# Below is the list of what's being queried using the search endpoint.
 
params = { 
    'assertion.code': ','.join(XBRL_Elements), 
    'report.filing-year': ','.join(report_year),
    'fields': ','.join(fields)
    }



### Execute the query with loop for all results 
### THIS SECTION DOES NOT NEED TO BE EDITED

search_endpoint = baseurl + 'api/v1/' + endpoint + '/search'
if unique:
    search_endpoint += "?unique"
orig_fields = params['fields']
offset_value = 0
res_df = []
count = 0
query_start = datetime.now()
printed = False
run_query = True

while True:
    if not printed:
        print("On", query_start.strftime("%c"), tokenInfo.email, "(client ID:", str(tokenInfo.client_id.split('-')[0]), "...) started the query and")
        printed = True
    retry = 0
    while retry < 3:
        res = requests.get(search_endpoint, params=params, headers={'Authorization' : 'Bearer {}'.format(tokenInfo.access_token)})
        res_json = res.json()
        if 'error' in res_json:
            if res_json['error_description'] == 'Bad or expired token':
                tokenInfo = refresh(tokenInfo)
            else: 
                print('There was an error: {}'.format(res_json['error_description']))
                run_query = False
                break
        else: 
		        break
        retry +=1
        if retry >= 3:
            print("Can't refresh the access token.  Run the first query block, then rerun the query.")
            run_query = False

    if not run_query:
       break

    print("up to", str(offset_value + res_json['paging']['limit']), "records are found so far ...")

    res_df += res_json['data']

    if res_json['paging']['count'] < res_json['paging']['limit']:
        print(" - this set contained fewer than the", res_json['paging']['limit'], "possible, only", str(res_json['paging']['count']), "records.")
        break
    else: 
        offset_value += res_json['paging']['limit'] 
        if 100 == res_json['paging']['limit']:
                params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)
                if offset_value == 10 * res_json['paging']['limit']:
                        break 
        elif 500 == res_json['paging']['limit']:
                params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)
                if offset_value == 4 * res_json['paging']['limit']:
                        break 
        params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)

if not 'error' in res_json:
    current_datetime = datetime.now().replace(microsecond=0)
    time_taken = current_datetime - query_start
    index = pd.DataFrame(res_df).index
    total_rows = len(index)
    your_limit = res_json['paging']['limit']
    limit_message = "If the results below match the limit noted above, you might not be seeing all rows, and should consider upgrading (https://xbrl.us/access-token).\n"
    
    if your_limit == 100:
        print("\nThis non-Member account has a limit of " , 10 * your_limit, " rows per query from our Public Filings Database. " + limit_message)
    elif your_limit == 500:
        print("\nThis Basic Individual Member account has a limit of ", 4 * your_limit, " rows per query from our Public Filings Database. " + limit_message)
    
    print("\nAt " + current_datetime.strftime("%c") +  ", the query finished with  ", str(total_rows), "  rows returned in " + str(time_taken) + " for \n" +  urllib.parse.unquote(res.url))
    
    df = pd.DataFrame(res_df)
    # the format truncates the HTML display of numerical values to two decimals; .csv data is unaffected
    pd.options.display.float_format = '{:,.2f}'.format
    display(HTML(df.to_html(max_rows=rows_to_display)))

On Sun Nov 24 08:35:09 2024 david.tauriello@xbrl.us (client ID: 328eae3e ...) started the query and
up to 5000 records are found so far ...
 - this set contained fewer than the 5000 possible, only 222 records.

At Sun Nov 24 08:35:11 2024, the query finished with   222   rows returned in 0:00:01.618231 for 
https://testapi.xbrl.us/api/v1/assertion/search?unique&assertion.code=DQC.US.0194.10621,DQC.US.0194.10636,DQC.US.0195.10622,DQC.US.0195.10623,DQC.US.0195.10624,DQC.US.0195.10625,DQC.US.0195.10626,DQC.US.0195.10627,DQC.US.0196.10628,DQC.US.0196.10629,DQC.US.0196.10630,DQC.US.0196.10631,DQC.US.0196.10632,DQC.US.0196.10633,DQC.US.0197.10634,DQC.US.0196.10635&report.filing-year=2023,2024&fields=report.entry-url,assertion.code.sort(ASC),report.base-taxonomy,report.document-type,assertion.run-date,report.accepted-timestamp.sort(DESC),report.accession,entity.code,entity.name,assertion.type,assertion.limit()


,report.entry-url,assertion.code,report.base-taxonomy,report.document-type,assertion.run-date,report.accepted-timestamp,report.accession,entity.code,entity.name,assertion.type
0,http://www.sec.gov/Archives/edgar/data/1862068/000182912623003671/rubicontech_10q.htm,DQC.US.0194.10621,US GAAP 2023,10-Q,2024-11-15,2023-05-22 17:06:00,0001829126-23-003671,0001862068,"Rubicon Technologies, Inc.",0194
1,http://www.sec.gov/Archives/edgar/data/66600/000149315223018572/form10-q.htm,DQC.US.0194.10621,US GAAP 2023,10-Q,2024-11-15,2023-05-22 16:55:00,0001493152-23-018572,0000066600,"Quad M Solutions, Inc",0194
2,http://www.sec.gov/Archives/edgar/data/1144546/000168316823003596/hfactor_i10q-033123.htm,DQC.US.0194.10621,US GAAP 2023,10-Q,2024-11-15,2023-05-22 09:29:00,0001683168-23-003596,0001144546,"HFactor, Inc.",0194
3,http://www.sec.gov/Archives/edgar/data/1021917/000149315223018272/form10-q.htm,DQC.US.0194.10621,US GAAP 2023,10-Q,2024-11-15,2023-05-19 12:35:00,0001493152-23-018272,0001021917,"AWAYSIS CAPITAL, INC.",0194
4,http://www.sec.gov/Archives/edgar/data/1812727/000149315223018163/form10-q.htm,DQC.US.0194.10621,US GAAP 2023,10-Q,2024-11-15,2023-05-18 17:28:00,0001493152-23-018163,0001812727,"RELIANCE GLOBAL GROUP, INC.",0194
5,http://www.sec.gov/Archives/edgar/data/1625288/000149315223017925/form10-q.htm,DQC.US.0194.10621,US GAAP 2023,10-Q,2024-11-15,2023-05-17 14:24:00,0001493152-23-017925,0001625288,"NEXIEN BIOPHARMA, INC.",0194
6,http://www.sec.gov/Archives/edgar/data/1619312/000182912623003452/lightstonevalue4_10q.htm,DQC.US.0194.10621,US GAAP 2023,10-Q,2024-11-15,2023-05-15 16:57:00,0001829126-23-003452,0001619312,"Lightstone Value Plus REIT IV, Inc.",0194
7,http://www.sec.gov/Archives/edgar/data/1360565/000149315223017088/form10-q.htm,DQC.US.0194.10621,US GAAP 2023,10-Q,2024-11-15,2023-05-15 12:23:00,0001493152-23-017088,0001360565,"WHERE FOOD COMES FROM, INC.",0194
8,http://www.sec.gov/Archives/edgar/data/1536089/000149315223017060/form10-q.htm,DQC.US.0194.10621,US GAAP 2023,10-Q,2024-11-15,2023-05-15 11:40:00,0001493152-23-017060,0001536089,VIRTUAL INTERACTIVE TECHNOLOGIES CORP.,0194
9,http://www.sec.gov/Archives/edgar/data/1713210/000149315223017030/form10-q.htm,DQC.US.0194.10621,US GAAP 2023,10-Q,2024-11-15,2023-05-15 10:38:00,0001493152-23-017030,0001713210,AGAPE ATP CORPORATION,0194


In [ ]:
# If you run this program locally, you can save the output to a file 
# on your computer (modify D:\results.csv to your system)

df.to_csv(r"D:\assertions-public-exposure.csv",sep=",")

# Google Colab users - comment out the line above and uncomment the code below to save the data frame as a .csv in your Google Drive

#from google.colab import drive
#drive.mount('drive')
#df.to_csv('assertions-public-exposure.csv')
#!cp data.csv "drive/My Drive/"